# Correction - Exercice 05 : régression logistique

Objectif : prédire si un étudiant valide un examen avec une régression logistique.

La cible est binaire :

- `0` : l'étudiant ne valide pas ;
- `1` : l'étudiant valide.

## 1. Charger le dataset

In [ ]:
import pandas as pd

data = pd.DataFrame({
    "heures_revision": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 15, 16, 18, 20],
    "presence": [0.35, 0.40, 0.45, 0.50, 0.55, 0.62, 0.65, 0.72, 0.76, 0.80, 0.84, 0.88, 0.90, 0.92, 0.95, 0.98],
    "controle_continu": [5, 6, 7, 8, 8, 9, 10, 11, 11, 12, 13, 13, 14, 15, 16, 17],
    "projet_rendu": [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    "valide": [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
})

data

## 2. Séparer X et y

`X` contient les variables explicatives.

`y` contient la cible à prédire.

In [ ]:
X = data[["heures_revision", "presence", "controle_continu", "projet_rendu"]]
y = data["valide"]

X.head()

In [ ]:
y.head()

## 3. Séparer train et test

On garde 25 % des données pour le test.

`random_state=42` permet d'obtenir le même découpage à chaque exécution.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

## 4. Entraîner une régression logistique

`LogisticRegression` apprend à relier les variables explicatives à une probabilité de validation.

In [ ]:
from sklearn.linear_model import LogisticRegression

modele = LogisticRegression(max_iter=1000)

modele.fit(X_train, y_train)

## 5. Prédire des classes

`predict` renvoie directement la classe prédite : `0` ou `1`.

In [ ]:
predictions = modele.predict(X_test)

predictions

## 6. Prédire des probabilités

`predict_proba` renvoie une probabilité pour chaque classe.

La colonne `proba_classe_1` correspond à la probabilité de validation.

In [ ]:
probabilites = modele.predict_proba(X_test)

pd.DataFrame(probabilites, columns=["proba_classe_0", "proba_classe_1"])

In [ ]:
resultats = X_test.copy()
resultats["classe_reelle"] = y_test
resultats["classe_predite"] = predictions
resultats["proba_validation"] = probabilites[:, 1]

resultats

## 7. Mesurer l'accuracy

L'accuracy mesure la proportion de prédictions correctes.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("Accuracy test :", accuracy)

## 8. Lire la matrice de confusion

La matrice de confusion montre les bonnes et mauvaises prédictions par classe.

In [ ]:
from sklearn.metrics import confusion_matrix

matrice = confusion_matrix(y_test, predictions)

pd.DataFrame(
    matrice,
    index=["réel_0", "réel_1"],
    columns=["prédit_0", "prédit_1"]
)

## 9. Tester un nouvel étudiant

On prédit maintenant la classe et la probabilité de validation d'un étudiant qui n'était pas dans le dataset.

In [ ]:
nouvel_etudiant = pd.DataFrame({
    "heures_revision": [9],
    "presence": [0.78],
    "controle_continu": [12],
    "projet_rendu": [1]
})

classe = modele.predict(nouvel_etudiant)[0]
proba = modele.predict_proba(nouvel_etudiant)[0, 1]

print("Classe prédite :", classe)
print("Probabilité de valider :", proba)

## Questions finales - Correction

### 1. Pourquoi la régression logistique est-elle utilisée pour une classification ?

Elle est utilisée pour une classification parce qu'elle prédit une probabilité entre `0` et `1`, puis transforme cette probabilité en classe avec un seuil.

Pour une classification binaire :

```text
p >= 0.5 -> classe 1
p < 0.5  -> classe 0
```

### 2. Quelle est la différence entre `predict` et `predict_proba` ?

`predict` donne directement la classe finale.

`predict_proba` donne les probabilités associées aux classes.

### 3. Que signifie une probabilité de `0.82` pour la classe `1` ?

Cela signifie que le modèle estime que l'étudiant a environ `82 %` de probabilité de valider.

### 4. À quoi sert le seuil de décision ?

Le seuil transforme une probabilité en classe.

Avec un seuil de `0.5`, une probabilité de `0.82` donne la classe `1`.

### 5. Pourquoi la matrice de confusion donne-t-elle plus d'information que l'accuracy seule ?

L'accuracy donne une proportion globale de bonnes réponses.

La matrice de confusion montre le détail : combien de `0` et de `1` ont été correctement ou incorrectement prédits.